# **ginger**

## **Setup**

In [ ]:
import sys
import os
import torch

def get_github_token():
    token = os.getenv("GITHUB_TOKEN")

    if token:
        return token

    import getpass
    return getpass.getpass("GitHub token: ")


try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
    print("Autoreload enabled.")
except Exception as e:
    print("Autoreload not available:", e)


if "google.colab" in sys.modules:
    print("Colab detected. Setting up repository...")

    %cd /content

    token = get_github_token()
    repo_url = f"https://{token}@github.com/Mango-Cats/ginger.git"

    if os.path.exists("ginger"):
        %cd ginger
        !git reset --hard HEAD
        !git pull origin main
    else:
        !git clone {repo_url}
        %cd ginger

    !pip install -e . --no-deps

else:
    print("Local environment detected.")

    if os.path.exists("ginger"):
        %cd ginger
    else:
        print("Repo not cloned locally. Set GITHUB_TOKEN or clone manually.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(DEVICE)

Autoreload not available: No module named 'imp'
Colab detected. Setting up repository...
/content
/content/ginger
HEAD is now at be20fea Fix error
From https://github.com/Mango-Cats/ginger
 * branch            main       -> FETCH_HEAD
Already up to date.
Obtaining file:///content/ginger
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ginger (pyproject.toml) ... done
  Created wheel for ginger: filename=ginger-0.1.0-0.editable-py3-none-any.whl size=5480 sha256=35bfa370bffaf4d6143001ce80d38119fa57c1e9255cf41656df53f4547325a3
  Stored in directory: /tmp/pip-ephem-wheel-cache-ubkkdjyc/wheels/4a/45/44/d3f200324dc86253cc07b97b9015a378520aae0b30f78eb83b
Successfully built ginger
  Attempting uninstall: ginger
    Found existing installation: ginger 0.1.0
    Uninstalling ginger-0.1.0:
      Successfully uninstal

## **Data Loading & Preprocessing**

In [29]:
from src.dataman import create_dataloaders
from src.dataman.constants import PAD_IDX
from pathlib import Path

In [30]:
DATA_PATH = Path("data/wikipron_tagalog.tsv")
BATCH_SIZE = 64

train_loader, val_loader, test_loader, grapheme_vocab, phoneme_vocab = create_dataloaders(
    DATA_PATH,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

print(f"Train samples: {len(train_loader.dataset)}")
print(f"Val samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")
print(f"Grapheme vocab size: {len(grapheme_vocab)}")
print(f"Phoneme vocab size: {len(phoneme_vocab)}")

Train samples: 14590
Val samples: 1824
Test samples: 1824
Grapheme vocab size: 32
Phoneme vocab size: 34


## **Model Initialization**

In [31]:
from src.architecture.transformer import Transformer
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim import AdamW
import torch.nn as nn

In [32]:
EMBEDDING_DIM = 128
NUM_ATTENTION_HEADS = 4
NUM_ENCODER_LAYERS = 4
NUM_DECODER_LAYERS = 4
FEEDFORWARD_DIM = 512
DROPOUT_RATE = 0.1
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 100 

model = Transformer(
    num_graphemes=len(grapheme_vocab),
    num_phonemes=len(phoneme_vocab),
    embedding_dim=EMBEDDING_DIM,
    num_attention_heads=NUM_ATTENTION_HEADS,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    feedforward_dim=FEEDFORWARD_DIM,
    dropout_rate=DROPOUT_RATE,
).to(DEVICE)

model = torch.compile(model)

CRITERION = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
OPTIMIZER = AdamW(params=model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
SCHEDULER = OneCycleLR(
    optimizer=OPTIMIZER,
    max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1 
)
SCALER = torch.amp.GradScaler(DEVICE)
PATIENCE = 7

## **Training**

In [33]:
from src.trainer import train_model

In [34]:
results = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=CRITERION,
    optimizer=OPTIMIZER,
    scheduler=SCHEDULER,
    scaler=SCALER,
    device=DEVICE,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
)


Starting training for 100 epochs...

Epoch   5/100 | Train Loss: 0.9902 | Val Loss: 1.6344 (No improvement for 2 epochs) | Best (last 5): 1.4399 @ epoch 3
Epoch  10/100 | Train Loss: 0.7518 | Val Loss: 1.4683 (No improvement for 4 epochs) | Best (last 5): 1.3718 @ epoch 6
Epoch  15/100 | Train Loss: 0.5782 | Val Loss: 1.0367 (Best) | Best (last 5): 1.0367 @ epoch 15
Epoch  20/100 | Train Loss: 0.4156 | Val Loss: 0.8381 (Best) | Best (last 5): 0.8381 @ epoch 20
Epoch  25/100 | Train Loss: 0.2984 | Val Loss: 0.7796 (No improvement for 1 epochs) | Best (last 5): 0.5794 @ epoch 24
Epoch  30/100 | Train Loss: 0.2400 | Val Loss: 0.5868 (No improvement for 2 epochs) | Best (last 5): 0.4993 @ epoch 28
Epoch  35/100 | Train Loss: 0.2005 | Val Loss: 0.5454 (No improvement for 1 epochs) | Best (last 5): 0.4926 @ epoch 34
Epoch  40/100 | Train Loss: 0.2210 | Val Loss: 0.5653 (No improvement for 6 epochs) | Best (last 5): 0.5124 @ epoch 36

Stopping early (patience=7)
Best validation loss: 0.4926


## **Inference**

In [35]:
from src.evaluation.evaluator import evaluate_model

In [38]:
eval_results = evaluate_model(
    model=model,
    test_loader=test_loader,
    grapheme_vocab=grapheme_vocab,
    phoneme_vocab=phoneme_vocab,
    device=DEVICE,
)


Metrics:
Total Words Tested: 1824
Phoneme Error Rate (PER): 8.00%
Word Error Rate (WER):    29.99%

Sample Predictions:

Word:     TANTSING
Expected: t a n t͡ʃ i ŋ
Output:   t a n t͡ʃ i ŋ

Word:     LIWASAN
Expected: l i w a s a n
Output:   l i w a s a n

Word:     UBAD
Expected: ʔ u b a d
Output:   ʔ u b a d

Word:     BUENVIAJE
Expected: b u e n b i a h e
Output:   b u e n v i a h e

Word:     MALAGKIT
Expected: m a l a ɡ k i t
Output:   m a ɡ l a k i t
